In [1]:
import polars as pl

In [20]:
# defining a function which cuts Meta- and Raw-Data files down to the interested SH rows only
# a whole folder for one year of one street_type will be processed 
# the naming of the orginal files from BASt.de must not be changed for the function to work!
# after taking a look into one Meta-Data file it can be assumed that "SH" has the "Landesnummer" = 1

# function input parameters:
# source is the (relative) path to the folder where the Meta- and Raw-Data files are stored
# target is the (relative) path to the folder where the new files should get stored (folder must exist already!)
# year: select the year from which the data will be processed
# street_type: select if the data is for "A" (Autobahn) or "B" (Bundesstraße)

def SH_Aggregation(source, target, year, street_type):

    source_location = source

    # iterating over all elements in the folder
    for month in range(1, 13):
        month_str = f"{month:02d}"

        # dynamically creating the path to a source-file
        data_path = f"{source_location}/Roh_{year}_{month_str}_{street_type}_S.csv"
        meta_path = f"{source_location}/Meta_{year}_{month_str}_{street_type}_S.csv"

        # dynamically creating the path where to save the new .csv-file
        SH_data_path = f"{target}/Roh_{year}_{month_str}_{street_type}_S.csv"
        SH_meta_path = f"{target}/Meta_{year}_{month_str}_{street_type}_S.csv"
    
        # checking the Meta-Data of each month if the assumption holds, that 1 is the "Landesnummer" for "SH"
        meta_df = pl.read_csv(meta_path, encoding = "latin1", separator = ";")
        non_correct_meta = (
            meta_df
            .filter((pl.col("Landesnummer") == 1) & (pl.col("Landeskuerzel") != "SH"))
            .height
        )
        if non_correct_meta == 0:
            print(f"File Meta_{year}_{month_str}_{street_type}_S.csv is giving '1' as the ID for 'SH'.")

            # cut Meta-Data files to rows for Schleswig-Holstein only and save as new .csv
            meta_df.filter(
                (pl.col("Landesnummer") == 1) & (pl.col("Landeskuerzel") == "SH")
            ).write_csv(SH_meta_path)
    
            print(f"File Meta_{year}_{month_str}_{street_type}_S.csv got processed and aggregated to only SH rows.")

            # cut Raw-Data files to rows for Schleswig-Holstein only and save as new .csv
            pl.scan_csv(data_path, separator = ";").filter(
                pl.col("Land") == 1
            ).sink_csv(SH_data_path)
    
            print(f"File Roh_{year}_{month_str}_{street_type}_S.csv got processed and aggregated to only SH rows.")

        # give a message if for one month the explained assumption doesn't hold
        else:
            print(f"----->Month {month_str} could not be aggregated for SH because the meta data is diffent as expected!")
        
    
    print("--- All processes done ---")

In [28]:
# example for calling the function with parametes 
SH_Aggregation("Roh_2025_B_S", "SH_2025_B_S", "2025", "B")

File Meta_2025_01_B_S.csv is giving '1' as the ID for 'SH'.
File Meta_2025_01_B_S.csv got processed and aggregated to only SH rows.
File Roh_2025_01_B_S.csv got processed and aggregated to only SH rows.
File Meta_2025_02_B_S.csv is giving '1' as the ID for 'SH'.
File Meta_2025_02_B_S.csv got processed and aggregated to only SH rows.
File Roh_2025_02_B_S.csv got processed and aggregated to only SH rows.
File Meta_2025_03_B_S.csv is giving '1' as the ID for 'SH'.
File Meta_2025_03_B_S.csv got processed and aggregated to only SH rows.
File Roh_2025_03_B_S.csv got processed and aggregated to only SH rows.
File Meta_2025_04_B_S.csv is giving '1' as the ID for 'SH'.
File Meta_2025_04_B_S.csv got processed and aggregated to only SH rows.
File Roh_2025_04_B_S.csv got processed and aggregated to only SH rows.
File Meta_2025_05_B_S.csv is giving '1' as the ID for 'SH'.
File Meta_2025_05_B_S.csv got processed and aggregated to only SH rows.
File Roh_2025_05_B_S.csv got processed and aggregated to